# 07 · Saída para o agente e o operador (v2)
**CRISP-DM:** Implantação (protótipo)

`consultar(recurso, t_registro)` responde, no instante em que o incidente é registrado:
1. **estado** do recurso nos 15 min anteriores (Normal / Residual) e o **regime esperado** para aquele contexto;
2. se Residual, o **indicador impactado** (erros de negócio 4xx, erros de sistema 5xx, latência p95 ou volume), com o valor observado e a faixa normal do contexto;
3. o **início estimado** (expansão retroativa: 15 → 30 → 60 min, depois hora a hora até 4 h);
4. como complemento, a **abrangência**: generalizado ou concentrado em um consumidor específico.

A saída é um JSON com um resumo em linguagem natural (`motivo_estatistico`), pensado para ser lido por um agente LLM ou diretamente pelo operador.

In [1]:
%run ./00_funcoes.ipynb

00_funcoes (v2) carregado · /home/wsl/gh/mestrado/claude9/v2


In [2]:
import pickle
with open(RESULTADOS / "04_calibracao.pkl", "rb") as f:
    calib = pickle.load(f)
CAL = {L: calib[(PROPOSTO, L)] for L in FAIXAS_L}
slots = pd.read_csv(RESULTADOS / "03_regime_por_contexto.csv")
cenarios = pd.read_csv(RESULTADOS / "05_cenarios.csv", parse_dates=["t_registro"])
decisoes = pd.read_parquet(RESULTADOS / "06_decisoes.parquet")
brutos = pd.read_parquet(RESULTADOS / "02_brutos_consolidados.parquet")
B = Brutos(brutos)

In [3]:
def natural(a, x):
    """Valor na unidade que o operador entende."""
    if a == "log_vol":
        return f"{np.expm1(x):,.0f} transações/h".replace(",", ".")
    if a == "log_lat_p95":
        return f"{np.expm1(x):.0f} ms"
    return f"{100 * x:.1f}%"


def consultar(recurso, t_registro, fatia=None):
    """Estado do recurso no instante do registro. ``fatia``: registros das 4 h anteriores (opcional, para cenários)."""
    t = pd.Timestamp(t_registro)
    f = fatia if fatia is not None else B.fatia(recurso, t - pd.Timedelta(minutes=H_MAX), t)
    per = {}
    for i, (lo, hi) in enumerate(BANDAS):
        o = agregar_periodo(f, t - pd.Timedelta(minutes=hi), t - pd.Timedelta(minutes=lo))
        o["recurso"] = recurso
        per[i] = o
    P = pd.DataFrame.from_dict(per, orient="index")
    residuais, vezes_b = [], []
    for i, (lo, hi) in enumerate(BANDAS):
        cal = CAL[min(hi - lo, 60)]
        res, ind, vz = decidir(P.loc[[i]], cal)
        residuais.append(bool(res[0]))
        vezes_b.append(vz.iloc[0])
    estado, z_lo, z_hi = estimar_inicio(residuais)
    p0 = P.loc[[0]]
    med, esc, q05, q95 = CAL[15]["modelo"].referencia(p0)
    ctx = (p0["tipo_dia"].iloc[0], int(p0["hora"].iloc[0]))
    reg = slots[(slots["recurso"] == recurso) & (slots["tipo_dia"] == ctx[0]) & (slots["hora"] == ctx[1])]["nome"]
    saida = {
        "recurso": rot(recurso),
        "registro_incidente": str(t),
        "periodo_decisao": f"{t - pd.Timedelta(minutes=15):%d/%m %H:%M} a {t:%H:%M}",
        "contexto": {"tipo_dia": NOMES_DIA[ctx[0]], "hora": f"{ctx[1]}h"},
        "regime_esperado": reg.iloc[0] if len(reg) else None,
        "estado": estado,
    }
    v0 = vezes_b[0]
    if estado == "Normal":
        saida["indicadores"] = {INDICADOR[a]: {"observado": natural(a, p0[a].iloc[0]),
                                               "faixa_normal_p5_p95": [natural(a, q05[a].iloc[0]), natural(a, q95[a].iloc[0])]}
                                for a in ATRIBUTOS if pd.notna(p0[a].iloc[0])}
        saida["motivo_estatistico"] = (
            f"Nos 15 min anteriores ao registro, o recurso {rot(recurso)} estava dentro do regime normal esperado para "
            f"{NOMES_DIA[ctx[0]]} às {ctx[1]}h: nenhum indicador passou do seu limite (maior: {v0.max():.2f}× o limite).")
        return saida
    evid = []
    for a in v0.sort_values(ascending=False).index:
        if not v0[a] > 1:
            break
        evid.append({"indicador": INDICADOR[a], "observado": natural(a, p0[a].iloc[0]),
                     "faixa_normal_p5_p95": [natural(a, q05[a].iloc[0]), natural(a, q95[a].iloc[0])],
                     "vezes_o_limite": round(float(v0[a]), 2)})
    if not evid:  # residual só em bandas anteriores não ocorre (estado vem da banda 0)
        evid = [{"indicador": INDICADOR[v0.idxmax()], "vezes_o_limite": round(float(v0.max()), 2)}]
    principal = evid[0]["indicador"]
    linha = P.loc[0]
    abr = abrangencia(linha, principal)
    fam = {"erros 4xx": "4xx", "erros 5xx": "5xx"}.get(principal)
    inicio = (f"entre {z_lo} e {z_hi} min antes do registro" if z_hi is not None
              else f"há mais de {z_lo} min (não voltou ao normal em 4 h)")
    saida.update({
        "indicador_impactado": principal,
        "evidencias": evid,
        "inicio_estimado": inicio,
        "evolucao_por_faixa": [{"faixa_min_antes": f"{lo}–{hi}", "estado": "Residual" if r else "Normal",
                                "maior_vezes_limite": round(float(v.max()), 2)} for (lo, hi), r, v in zip(BANDAS, residuais, vezes_b)],
        "abrangencia": None if abr is None else {
            "classificacao": abr, "clientes_ativos": int(linha["n_cli"]),
            "clientes_com_erro": int(linha[f"n_cli_{fam}"]),
            "participacao_do_principal_cliente_nos_erros": f"{linha[f'top_err_{fam}']:.0%}",
            "participacao_desse_cliente_no_volume": f"{linha[f'top_vol_{fam}']:.0%}"},
    })
    e0 = evid[0]
    texto = (f"O recurso {rot(recurso)} está fora do regime normal esperado para {NOMES_DIA[ctx[0]]} às {ctx[1]}h. "
             f"Indicador impactado: {principal} — {e0.get('observado', '')} nos 15 min anteriores ao registro "
             f"(faixa normal {e0['faixa_normal_p5_p95'][0]} a {e0['faixa_normal_p5_p95'][1]}), "
             f"{e0['vezes_o_limite']}× o limite do recurso. O desvio começou {inicio}.")
    if abr == "consumidor específico":
        texto += f" Os erros estão concentrados em um consumidor ({linha[f'top_err_{fam}']:.0%} dos erros)."
    elif abr == "generalizado":
        texto += f" Os erros aparecem em {int(linha[f'n_cli_{fam}'])} de {int(linha['n_cli'])} consumidores ativos."
    saida["motivo_estatistico"] = texto
    return saida


def fatia_cenario(cen, semente=SEMENTE):
    """Mesma fatia com falha usada na avaliação (mesma semente por cenário)."""
    rng = np.random.default_rng(semente * 1_000_003 + int(cen.cenario))
    t = pd.Timestamp(cen.t_registro)
    f = B.fatia(cen.recurso, t - pd.Timedelta(minutes=H_MAX), t)
    if cen.tipo == "incidente":
        f = aplicar_motivo(f, t - pd.Timedelta(minutes=int(cen.atraso)), cen.motivo, cen.intensidade, rng)
    return f

## Estudos de caso
1. Erro de sistema (5xx) generalizado, iniciado 60 min antes do registro;
2. Erro de negócio (4xx) de um consumidor, iniciado 30 min antes;
3. Latência elevada, iniciada 120 min antes;
4. Consulta de controle (sem falha);
5. Evento **real** no teste (sem injeção): E02, madrugada de 02/09, quando 5xx atingiu a maioria dos consumidores.

In [4]:
dp = decisoes.loc[f"{PROPOSTO} | 15"]


def escolhe(motivo, atraso, intensidade="media"):
    c = cenarios[(cenarios["motivo"] == motivo) & (cenarios["atraso"] == atraso) & (cenarios["intensidade"] == intensidade)]
    ok = dp.index[dp["residual"] & (dp["indicador"] == dp["esperado"])]  # caso ilustrativo com indicador correto
    c = c[c["cenario"].isin(ok)]
    return c


casos = {}
for nome, (mot, atr) in {"5xx_generalizado": ("5xx_geral", 60), "4xx_um_consumidor": ("4xx_cliente", 30),
                         "latencia": ("latencia", 120)}.items():
    abr_esp = MOTIVOS[mot][1]
    for cen in escolhe(mot, atr).itertuples():  # ilustração: primeiro caso com indicador e abrangência corretos
        out = consultar(cen.recurso, cen.t_registro, fatia_cenario(cen))
        if mot == "latencia" or (out.get("abrangencia") or {}).get("classificacao") == abr_esp:
            break
    out["_gabarito"] = {"motivo": NOMES_MOTIVO[mot], "intensidade": cen.intensidade, "inicio_real_min_antes": int(atr)}
    casos[nome] = out
ctl = cenarios[(cenarios["tipo"] == "controle") & cenarios["cenario"].isin(dp.index[~dp["residual"]])].iloc[0]
casos["controle"] = consultar(ctl.recurso, ctl.t_registro)
E02 = [r for r in B.por_recurso if rot(r) == "E02"][0]
casos["evento_real_E02"] = consultar(E02, pd.Timestamp("2026-09-02 01:45"))

for nome, out in casos.items():
    print(f"===== {nome} =====")
    print(json.dumps(out, ensure_ascii=False, indent=2, default=str))

===== 5xx_generalizado =====
{
  "recurso": "E08",
  "registro_incidente": "2026-09-04 10:47:00",
  "periodo_decisao": "04/09 10:32 a 10:47",
  "contexto": {
    "tipo_dia": "dia útil",
    "hora": "10h"
  },
  "regime_esperado": "R4: volume médio, 4xx alto · dia útil · 7h, 10h",
  "estado": "Residual",
  "indicador_impactado": "erros 5xx",
  "evidencias": [
    {
      "indicador": "erros 5xx",
      "observado": "17.9%",
      "faixa_normal_p5_p95": [
        "0.0%",
        "0.1%"
      ],
      "vezes_o_limite": 5.98
    }
  ],
  "inicio_estimado": "entre 30 e 120 min antes do registro",
  "evolucao_por_faixa": [
    {
      "faixa_min_antes": "0–15",
      "estado": "Residual",
      "maior_vezes_limite": 5.98
    },
    {
      "faixa_min_antes": "15–30",
      "estado": "Residual",
      "maior_vezes_limite": 4.0
    },
    {
      "faixa_min_antes": "30–60",
      "estado": "Residual",
      "maior_vezes_limite": 3.76
    },
    {
      "faixa_min_antes": "60–120",
      "estad

In [5]:
salvar_json(casos, "07_estudos_de_caso.json")

## Plano de uso
- **Integração:** o agente de triagem recebe o incidente (recursos envolvidos e horário de registro) e chama `consultar` para cada recurso. `estado` e `indicador_impactado` orientam a análise; `evidencias` e `motivo_estatistico` sustentam a explicação; `abrangencia` é contexto adicional.
- **Retreino:** mensal, com o último mês completo (perfil de 1 h, limites por indicador e faixa por validação cruzada).
- **Recurso novo:** sem ao menos 2–3 semanas de histórico, a saída deve ser `indeterminado`.
- **Calendário:** feriados na lista `FERIADOS` (tratados como dia não útil).
- **Limitação:** manutenções não recorrentes aparecem como Residual; o operador deve cruzar com o calendário de mudanças.